# KOs enriched in certain sub-cell types


In [ ]:
from libraries import *
from parameters import *


In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
adata = sc.read("./Data/adataALL.h5ad")

In [ ]:
adata

In [ ]:
adata = adata[(adata.obs["guide_num"] == 1),]

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color="sample_name", size=0.5,
           color_map="coolwarm", vmax=2.0)


In [ ]:
adata.obs

In [ ]:
mtTable = pd.DataFrame(pd.crosstab(adata.obs["leiden"], adata.obs["guide_id"]))

In [ ]:
%%R  -i mtTable

rownames(mtTable) = paste0("TC_", rownames(mtTable))


myRes = list()

lappend <- function(lst, obj) {
  lst[[length(lst)+1]] <- obj
  return(lst)
}


totalCountsPerSCT <- data.frame(cluster = rownames(mtTable),
                                cellCounts = rowSums(mtTable))
rownames(totalCountsPerSCT) = totalCountsPerSCT$cluster
totalNoOfCells = sum(totalCountsPerSCT$cellCounts)


totalCountsPerKO <- data.frame(KO = colnames(mtTable),
                              cellCounts = colSums(mtTable))
    

depletedGuides <- list()

for (myKO in colnames(mtTable)){
    
    myDFKO = list()
    
    if(totalCountsPerKO[myKO, "cellCounts"] == 0){
        depletedGuides <- lappend(depletedGuides, myKO)
    }else{
        
    
        for(cluster in rownames(mtTable)){
           
           myDF = data.frame(KOcells = c(mtTable[cluster, myKO], 
                                         totalCountsPerKO[myKO, "cellCounts"] - mtTable[cluster, myKO]),
                             restNo = c(totalCountsPerSCT[cluster,"cellCounts"], 
                                        totalNoOfCells - totalCountsPerSCT[cluster,"cellCounts"]))
        
           #print(myDF)
           testRes_greater <- fisher.test(x= myDF, 
                                    alternative = "greater")
          # print(testRes_g)
            
           myDFKO = lappend(myDFKO, c("KO"=myKO,
                                    "cluster"=cluster, 
                                    "pval"=testRes_greater["p.value"],
                                    "estimate"=testRes_greater["estimate"],
                                    "direction"="greater"))
                        
           testRes_less <- fisher.test(x= myDF, alternative = "less")
           #print(testRes_l)
          
           myDFKO = lappend(myDFKO, c("KO"=myKO, 
                                    "cluster"=cluster, 
                                    "pval"=testRes_less["p.value"],
                                    "estimate"=testRes_less["estimate"],
                                    "direction"="less"))
           
                        
        }   
    }
    
    myDFKO = data.frame(do.call(rbind,myDFKO))
    myDFKO$FDR = p.adjust(myDFKO$pval)
    
    myRes = lappend(myRes, myDFKO)
}

myRes = data.frame(do.call(rbind,myRes))
colnames(myRes) = c("KO", "cluster", "Pvalue", "Estimate", "Direction", "FDR")

myRes=data.frame(lapply(myRes,unlist))


In [ ]:
%%R

myResSgn = myRes[myRes$FDR < 0.1,]
myResSgn$cluster = factor(myResSgn$cluster, levels=rownames(mtTable))
myResSgn_less = myResSgn[myResSgn$Direction == "less",]
myResSgn_greater = myResSgn[myResSgn$Direction == "greater",]

myResSgn_greater = myResSgn_greater[order(myResSgn_greater$cluster, myResSgn_greater$Pvalue),]
myResSgn_less = myResSgn_less[order(myResSgn_less$cluster, myResSgn_less$Pvalue),]

allRes = rbind(myResSgn_greater,myResSgn_less)

write.csv(allRes, "KOEnrichmentAndDepletion.csv", row.names=FALSE)

In [ ]:
%%R

allRes




In [ ]:
myRes_greaterSign

In [ ]:
%%R

myRes_less = myRes[myRes$Direction == "less",]
myRes_greater = myRes[myRes$Direction == "greater",]

myRes_less$FDR = 0
myRes_greater$FDR = 0

for(elem in rownames(mtTable)){
    print(elem)
    myRes_less[myRes_less$cluster == elem,"FDR"] = unlist(p.adjust(myRes_less[myRes_less$cluster == elem,"Pvalue"]))
    myRes_greater[myRes_greater$cluster == elem,"FDR"] = unlist(p.adjust(myRes_greater[myRes_less$cluster == elem,"Pvalue"]))

}

# myRes_less
# myRes_less$FDR = p.adjust(myRes_less$Pvalue, n=length(colnames(mtTable)) )
# myRes_greater$FDR = p.adjust(myRes_greater$Pvalue, n=length(colnames(mtTable))  )
#write.csv(myRes_greater, "KOsEnriched.csv")


In [ ]:
%%R
rbind(myRes_greater[myRes_greater$FDR != 1,],myRes_less[myRes_less$FDR != 1,])
      

In [ ]:
%%R
myRes_less[myRes_less$FDR != 1,]

In [ ]:
%%R




In [ ]:
%%R
myRes_greater_wide

In [ ]:
%%R  -i mtTable

lappend <- function(lst, obj) {
  lst[[length(lst)+1]] <- obj
  return(lst)
}

totalCountsPerSCT <- data.frame(cluster = paste0("TC_", seq(0,19,1)),
                                cellCounts = colSums(mtTable))


allGuidesPerSCT = data.frame(matrix(20, ncol(allGuidesPerSCT)))
rownames(allGuidesPerSCT)

allGuidesPerSCT$KOGuide <- as.character(allGuidesPerSCT$KOGuide)
rownames(allGuidesPerSCT) <- allGuidesPerSCT$KOGuide

for(i in seq(0,19,1)){
    allGuidesPerSCT[paste0(i,"_pval_greater")] = -1
    allGuidesPerSCT[paste0(i,"_estimate_greater")] = -1
    allGuidesPerSCT[paste0(i,"_pval_less")] = -1
    allGuidesPerSCT[paste0(i,"_estimate_less")] = -1
    
}



rownames(totalCountsPerSCT) = totalCountsPerSCT$subCelltype
totalNoOfCells = sum(totalCountsPerSCT$cellCounts)


depletedGuides <- list()

for (elem in as.character(unique(allGuidesPerSCT$KOGuide))){
    
    
    if(allGuidesPerSCT[elem, "noOfGuideCells"] == 0){
        depletedGuides <- lappend(depletedGuides, elem)
    }else{
        
        for(subCellType in c("DC1", "DC2", "MReg", "MacDC")){
           myDF = data.frame(clusterNo = c(allGuidesPerSCT[elem, subCellType], allGuidesPerSCT[elem, "noOfGuideCells"] - allGuidesPerSCT[elem, subCellType]),
                  restNo = c(totalCountsPerSCT[subCellType,"cellCounts"], totalNoOfCells - totalCountsPerSCT[subCellType,"cellCounts"]))
        
           #print(myDF)
           testRes_g <- fisher.test(x= myDF, alternative = "greater")
           #print(testRes_g)
           allGuidesPerSCT[elem, paste0(subCellType,"_pval_greater")] = testRes_g$p.value
           allGuidesPerSCT[elem, paste0(subCellType,"_estimate_greater")] = testRes_g$estimate
                        
           testRes_l <- fisher.test(x= myDF, alternative = "less")
           #print(testRes_l)
           allGuidesPerSCT[elem, paste0(subCellType,"_pval_less")] = testRes_l$p.value
           allGuidesPerSCT[elem, paste0(subCellType,"_estimate_less")] = testRes_l$estimate
                        
        }
        
    }  
}

for(i in c("DC1", "DC2", "MReg", "MacDC")){
    allGuidesPerSCT[paste0(i,"_FDR_greater")] <- p.adjust(allGuidesPerSCT[[paste0(i,"_pval_greater")]])
    allGuidesPerSCT[paste0(i,"_FDR_less")] <- p.adjust(allGuidesPerSCT[[paste0(i,"_pval_less")]])  
}


#saveRDS(allGuidesPerSCT, "/home/eraslab1/Projects/E3Ligase/analysisSingle/outputs/RDSFiles/allGuidesPerSCT_withTests_geneLevel.rds")


In [ ]:
%%R 
head(allGuidesPerSCT)

In [ ]:
%%R -w 2.5 -h 14 -u in

library("reshape2")


allGuidesPerSCT_FDRs <- allGuidesPerSCT[,c("DC1_FDR_greater", "DC2_FDR_greater","MReg_FDR_greater", "MacDC_FDR_greater",
                                          "DC1_FDR_less", "DC2_FDR_less", "MReg_FDR_less", "MacDC_FDR_less", "KOGuide")]

allGuidesPerSCT_FDRs_melted <- melt(allGuidesPerSCT_FDRs, id.vars="KOGuide")
allGuidesPerSCT_FDRs_melted <- allGuidesPerSCT_FDRs_melted[allGuidesPerSCT_FDRs_melted$value < 0.15,]

allGuidesPerSCT_FDRs_melted$celltype <- sapply(allGuidesPerSCT_FDRs_melted$variable,
                                               function(x){strsplit(as.character(x),"_")[[1]][1]})
allGuidesPerSCT_FDRs_melted$testType <- sapply(allGuidesPerSCT_FDRs_melted$variable,
                                               function(x){strsplit(as.character(x),"_")[[1]][3]})

allGuidesPerSCT_FDRs_melted$oddRatio = 1

for(i in 1:nrow(allGuidesPerSCT_FDRs_melted)){
    allGuidesPerSCT_FDRs_melted[i,"oddRatio"] = allGuidesPerSCT[allGuidesPerSCT_FDRs_melted[i,"KOGuide"], 
                                                                paste0(allGuidesPerSCT_FDRs_melted[i,"celltype"], "_estimate_", allGuidesPerSCT_FDRs_melted[i,"testType"])]
}   


kk <- data.frame(matrix(1.0, nrow= 4, ncol=length(unique(allGuidesPerSCT_FDRs_melted$KOGuide))))
colnames(kk) = unique(allGuidesPerSCT_FDRs_melted$KOGuide)
rownames(kk) = c("DC1", "DC2", "MReg", "MacDC")


for(i in 1:nrow(allGuidesPerSCT_FDRs_melted)){
        kk[allGuidesPerSCT_FDRs_melted[i, "celltype"],  allGuidesPerSCT_FDRs_melted[i, "KOGuide"]] = allGuidesPerSCT_FDRs_melted[i, "oddRatio"]
}

dim(kk)

kk[kk > 2] = 2

head(kk)
#kk = t(kk)

kk <- kk[c("DC2", "MacDC", "MReg", "DC1"),]
kk <- data.frame(t(kk))
kk = kk[order(-kk$DC2, -kk$MacDC, -kk$MReg, -kk$DC1),]
library(pheatmap)

head(kk)
rownames(kk) = sapply(rownames(kk), function(x){strsplit(x,"_")[[1]][2]})
kk = kk[rownames(kk) != "NO",]
# write.csv(rownames(kk), "EnrichedDepletedGuides.csv")
pheatmap(kk,cluster_rows=F, cluster_cols=F, treeheight_col=0, treeheight_row=0, color=hcl.colors(50, "Tropic"), clustering_method="ward.D2")